### Classification example inference with Ryzen AI

This example demonstrates the 5 steps of classification model inference on the embedded Neural Processing Unit (NPU) in your AMD Ryzen AI enabled PC. The steps are as follows:

The notebook runs unchanged on Windows (Ryzen AI conda environment) and Linux (Ryzen AI venv). All OS-specific handling is done in the **Platform Setup** cell below.

In [ ]:
# Import necessary libraries
import os
import torch
import torch.nn as nn
import torchvision
import subprocess
import onnxruntime
import numpy as np
import onnx
import shutil
import time 
from timeit import default_timer as timer
from quark.onnx import ModelQuantizer  
from quark.onnx.quantization.config import Config, get_default_config  
from utils_custom import ImageDataReader, evaluate_onnx_model 
import json  
import shutil
import sys

#### Platform Setup

This cell detects the operating system and defines the helpers used by the NPU section: the Ryzen AI installation path, the NPU type (`pnputil` on Windows, `lspci` on Linux) and the matching VitisAI Execution Provider options. It also checks that the notebook kernel is the Ryzen AI environment.

In [ ]:
# ---------------- Platform Setup ---------------- #
import hashlib

IS_WINDOWS = sys.platform == "win32"
IS_LINUX = sys.platform.startswith("linux")


def get_install_dir():
    """Ryzen AI install path: set by the installer on Windows, by <TARGET-PATH>/venv/bin/activate on Linux."""
    install_dir = os.environ.get("RYZEN_AI_INSTALLATION_PATH")
    if not install_dir:
        if IS_LINUX:
            hint = ("Select the 'Ryzen AI 1.7.1 (Linux)' kernel, or start Jupyter from a shell where "
                    "/opt/xilinx/xrt/setup.sh and <TARGET-PATH>/venv/bin/activate have been sourced (see README).")
        else:
            hint = r"Set it in the NPU Inference section below, e.g. C:\Program Files\RyzenAI\<version>."
        raise RuntimeError(f"RYZEN_AI_INSTALLATION_PATH is not set. {hint}")
    return install_dir


def get_npu_type():
    """Return 'PHX/HPT', 'STX' or 'KRK' on Windows, 'STX/KRK' or 'PHX/HPT' on Linux, '' if none is found."""
    if IS_WINDOWS:
        out = subprocess.run("pnputil /enum-devices /bus PCI /deviceids", shell=True,
                             capture_output=True, text=True, errors="ignore").stdout
        if "PCI\\VEN_1022&DEV_1502&REV_00" in out:
            return "PHX/HPT"
        if "PCI\\VEN_1022&DEV_17F0&REV_20" in out:
            return "KRK"
        if any(f"PCI\\VEN_1022&DEV_17F0&REV_{rev}" in out for rev in ("00", "10", "11")):
            return "STX"
    elif IS_LINUX:
        try:
            out = subprocess.run(["lspci", "-nn"], capture_output=True, text=True).stdout
        except FileNotFoundError:
            raise RuntimeError("lspci not found. Install it with: sudo apt install pciutils")
        if "1022:17f0" in out:
            return "STX/KRK"
        if "1022:1502" in out:
            return "PHX/HPT"
    return ""


def get_npu_provider_options(npu_type, install_dir, model_path, cache_dir):
    """VitisAI Execution Provider options for the detected NPU."""
    if npu_type not in ("PHX/HPT", "STX", "KRK", "STX/KRK"):
        raise RuntimeError(f"No supported Ryzen AI NPU detected on {sys.platform}.")
    if npu_type == "PHX/HPT" and not IS_WINDOWS:
        raise RuntimeError("PHX/HPT NPUs are not supported by Ryzen AI 1.7.1 on Linux (STX/KRK only).")

    # The EP reuses a compiled model from cache_dir by cache_key, and a cache must not be reused
    # across Vitis AI EP versions, so the key must change with both the model and the EP version.
    with open(model_path, "rb") as f:
        model_hash = hashlib.sha256(f.read() + onnxruntime.__version__.encode()).hexdigest()[:12]
    options = {
        "cache_dir": str(cache_dir),
        "cache_key": f"{os.path.splitext(os.path.basename(model_path))[0]}_{model_hash}",
        "enable_cache_file_io_in_mem": 0,  # INT8 models are only written to cache_dir when this is 0
        "ai_analyzer_visualization": True,
        "ai_analyzer_profiling": True,
    }
    if npu_type == "PHX/HPT":
        options["target"] = "X1"
        options["xclbin"] = os.path.join(install_dir, "voe-4.0-win_amd64", "xclbins", "phoenix", "4x4.xclbin")
    return [options]


available_providers = onnxruntime.get_available_providers()
if "VitisAIExecutionProvider" not in available_providers:
    env_hint = ("the 'Ryzen AI 1.7.1 (Linux)' kernel (see README)" if IS_LINUX
                else "the Ryzen AI conda environment as the notebook kernel")
    raise RuntimeError(f"VitisAIExecutionProvider is not available. Select {env_hint}.")
if IS_LINUX:
    # Without the activated venv the NPU libraries cannot be loaded, so fail here rather than at session creation.
    get_install_dir()

print(f"Platform: {sys.platform}")
print(f"ONNX Runtime providers: {available_providers}")

#### 1. Get Model

Here, we'll use the resnet50 model as an example. You may choose any classification models train with Imagenet from torchvision.

In [ ]:
# ---------------- Model Setup ---------------- #

# Define directories
models_dir = "models"
os.makedirs(models_dir, exist_ok=True)

# Load pre-trained ResNet50 model
model = torchvision.models.resnet50(weights="IMAGENET1K_V2")

# Save the model
model.to("cpu")
torch.save(model, os.path.join(models_dir, "resnet50.pt"))


#### 2. Export to ONNX

The model inference with Ryzen AI is based on onnxruntime. The following code is used for exporting a PyTorch model to the ONNX (Open Neural Network Exchange) format. The ONNX file is needed to use the AMD Quark Quantizer.

In [ ]:
# Export model to ONNX
dummy_inputs = torch.randn(1, 3, 224, 224)
input_names = ['input']
output_names = ['output']
dynamic_axes = {'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
tmp_model_path = os.path.join(models_dir, "resnet50.onnx")

torch.onnx.export(
    model,
    dummy_inputs,
    tmp_model_path,
    export_params=True,
    opset_version=17,  # Recommended opset
    input_names=input_names,
    output_names=output_names,
    dynamic_axes=dynamic_axes,
)

print(f"✅ Model exported to ONNX at: {tmp_model_path}")

#### 3. Quantize Model

Using the AMD Quark Quantizer and providing the newly exported ONNX model, we'll quantize the model. The quantization progress will need the calibration data from Imagenet. Download the data from [here](https://huggingface.co/datasets/ILSVRC/imagenet-1k/tree/script/data) to download it.
You need to register on Hugging Face and download the following file:
**val_images.tar.gz**.
This file contains a subset of ImageNet images used specifically for calibration.

The dataset is gated, so the download needs a Hugging Face access token. See the README section *Download the Calibration Dataset* for scripted downloads with `hf download`, `wget` or `curl`. Save the file as `data/val_images.tar.gz` and extract it into a `val_images` folder. The same two commands work in a Linux shell and in a Windows 10+ Command Prompt or PowerShell:

```
mkdir val_images
tar -xzf data/val_images.tar.gz -C val_images
```

The code below reads the images from the `val_images` folder and creates a `calib_data` folder. It is skipped if `calib_data` already exists.

In [ ]:
#
# Copyright (C) 2023, Advanced Micro Devices, Inc. All rights reserved.
# SPDX-License-Identifier: MIT
#

source_folder = 'val_images'
calib_data_path = 'calib_data'

if os.path.isdir(calib_data_path) and os.listdir(calib_data_path):
    print(f"'{calib_data_path}' already exists, skipping dataset preparation.")
else:
    if not os.path.isdir(source_folder):
        raise FileNotFoundError(
            f"'{source_folder}' not found. Download val_images.tar.gz and extract it as described above.")

    for filename in sorted(os.listdir(source_folder)):
        if not filename.startswith('ILSVRC2012_val_') or not filename.endswith('.JPEG'):
            continue

        n_identifier = filename.split('_')[-1].split('.')[0]
        folder_path = os.path.join(source_folder, n_identifier)
        os.makedirs(folder_path, exist_ok=True)
        shutil.move(os.path.join(source_folder, filename), os.path.join(folder_path, filename))

    print("File organization complete.")

    os.makedirs(calib_data_path, exist_ok=True)

    for subfolder in sorted(os.listdir(source_folder)):
        source_subfolder = os.path.join(source_folder, subfolder)
        if not os.path.isdir(source_subfolder):
            continue
        destination_subfolder = os.path.join(calib_data_path, subfolder)
        os.makedirs(destination_subfolder, exist_ok=True)

        files = sorted(os.listdir(source_subfolder))
        if files:
            shutil.copy(os.path.join(source_subfolder, files[0]), os.path.join(destination_subfolder, files[0]))

    print("Creating calibration dataset complete.")

In [ ]:
# ---------------- Quark Quantization ---------------- #

# Define dataset directory
calib_dir = "calib_data" 

# Set input & output ONNX model paths
input_model_path = tmp_model_path
output_model_path = os.path.join(models_dir, "resnet50_quantized.onnx")

# Preprocessing transformations
preprocess = torchvision.transforms.Compose([
    torchvision.transforms.Resize(256),
    torchvision.transforms.CenterCrop(224),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load dataset
calib_dataset = torchvision.datasets.ImageFolder(root=calib_dir, transform=preprocess)

#Data set 
num_calib_data = 54  
calib_dataset = torch.utils.data.Subset(calib_dataset, range(num_calib_data))

# Define DataLoader for Calibration
calibration_dataloader = torch.utils.data.DataLoader(calib_dataset, batch_size=6, shuffle=False)

# Configure Quark Quantization
quant_config = get_default_config("XINT8")  # Use XINT8 quantization  
config = Config(global_quant_config=quant_config)

# Create an ONNX Quantizer  
quantizer = ModelQuantizer(config)  

# Perform Quark Quantization  
quant_model = quantizer.quantize_model(
    model_input=input_model_path,   
    model_output=output_model_path,   
    calibration_data_reader=ImageDataReader(calibration_dataloader)  # Use ImageDataReader from utils_custom
)

print(f"✅ Quark Quantized model saved at: {output_model_path}")

#### 4. Model inference on CPU / iGPU / NPU with single image

Now we have successfully quantized the model, and we will use the onnxruntime to do the inference on CPU, iGPU and NPU. The iGPU path uses DirectML, which is only available on Windows; on Linux that cell is skipped.

In [ ]:
from PIL import Image

def load_labels(path):
    with open(path) as f:
        data = json.load(f)
    return np.asarray(data)

def preprocess_image(input):
    normalize = torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
  
    transform = torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Resize((224, 224)),
        normalize,
    ])
    img_tensor = transform(input).unsqueeze(0)
    return img_tensor.numpy()

def softmax(x):
    x = x.reshape(-1)
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)

def postprocess(result):
    return softmax(np.array(result)).tolist()

labels = load_labels('data/imagenet-simple-labels.json')
image = Image.open('data/dog.jpg')

print("Image size: ", image.size)
input_data = preprocess_image(image)

#### CPU Inference

In [ ]:
# Run inference on CPU
onnx_model_path = output_model_path
cpu_options = onnxruntime.SessionOptions()

cpu_session = onnxruntime.InferenceSession(
    onnx_model_path,
    providers=['CPUExecutionProvider'],
    sess_options=cpu_options,
)

start = timer()
cpu_outputs = cpu_session.run(None, {'input': input_data})
end = timer()

cpu_results = postprocess(cpu_outputs)
inference_time = np.round((end - start) * 1000, 2)
idx = np.argmax(cpu_results)

print('----------------------------------------')
print(f'Final top prediction is: {labels[idx]}')
print('----------------------------------------')
print(f'Inference time: {inference_time} ms')
print('----------------------------------------')

sort_idx = np.flip(np.squeeze(np.argsort(cpu_results)))
print('------------ Top 5 labels are: ----------------------------')
print(labels[sort_idx[:5]])
print('-----------------------------------------------------------')

#### iGPU Inference (Windows only)

DirectML (`DmlExecutionProvider`) ships only with the Windows Ryzen AI environment. On other platforms the cell prints a message and skips.

In [ ]:
#iGPU inference
if "DmlExecutionProvider" not in onnxruntime.get_available_providers():
    print("DirectML (iGPU) is not available on this platform, skipping iGPU inference.")
else:
    dml_options = onnxruntime.SessionOptions()

    # Create Inference Session to run the quantized model on the iGPU
    dml_session = onnxruntime.InferenceSession(
        onnx_model_path,
        providers = ['DmlExecutionProvider'],
        provider_options = [{"device_id": "0"}]
    )
    start = time.time()
    dml_outputs = dml_session.run(None, {'input': input_data})
    end = time.time()

    dml_results = postprocess(dml_outputs)
    inference_time = np.round((end - start) * 1000, 2)
    idx = np.argmax(dml_results)

    print('----------------------------------------')
    print('Final top prediction is: ' + labels[idx])
    print('----------------------------------------')

    print('----------------------------------------')
    print('Inference time: ' + str(inference_time) + " ms")
    print('----------------------------------------')

    sort_idx = np.flip(np.squeeze(np.argsort(dml_results)))
    print('------------ Top 5 labels are: ----------------------------')
    print(labels[sort_idx[:5]])
    print('-----------------------------------------------------------')

#### NPU Inference

In [ ]:
# RYZEN_AI_INSTALLATION_PATH is set by the Ryzen AI installer on Windows and by
# <TARGET-PATH>/venv/bin/activate on Linux. If it is missing, set it here, for example:
# os.environ['RYZEN_AI_INSTALLATION_PATH'] = r'C:\Program Files\RyzenAI\1.7.1'   # Windows
# On Linux, setting the variable is not enough: start the kernel from the activated venv (see README).
install_dir = get_install_dir()
print(f"Ryzen AI installation path: {install_dir}")

In [ ]:
#NPU inference

# Detect the NPU and build the matching provider options (see the Platform Setup cell)
npu_type = get_npu_type()
print(f"NPU Type: {npu_type}")

cache_dir = os.path.join(os.getcwd(), "cache")
provider_options = get_npu_provider_options(npu_type, install_dir, onnx_model_path, cache_dir)

# Create session options
session_options = onnxruntime.SessionOptions()
# 0=Verbose, 1=Info, 2=Warning, 3=Error, 4=Fatal. Verbose/Info flood stderr during NPU compilation,
# which can hang the Jupyter kernel on Linux; use them only when running as a script.
session_options.log_severity_level = 3

npu_session = onnxruntime.InferenceSession(
    onnx_model_path,
    sess_options = session_options,
    providers = ['VitisAIExecutionProvider'],
    provider_options = provider_options
)

start = time.time()
npu_outputs = npu_session.run(None, {'input': input_data})
end = time.time()

npu_results = postprocess(npu_outputs)
inference_time = np.round((end - start) * 1000, 2)
idx = np.argmax(npu_results)

print('----------------------------------------')
print('Final top prediction is: ' + labels[idx])
print('----------------------------------------')

print('----------------------------------------')
print('Inference time: ' + str(inference_time) + " ms")
print('----------------------------------------')

sort_idx = np.flip(np.squeeze(np.argsort(npu_results)))
print('------------ Top 5 labels are: ----------------------------')
print(labels[sort_idx[:5]])
print('-----------------------------------------------------------')

#### 5. Model Analysis on NPU

After NPU inference, there are several '.json' files generated by the Ryzen AI tracing tool, which could be open by the AI Analyzer for further optimization.

This step is optional. `aianalyzer` is included in the Ryzen AI environment on both Windows and Linux; it serves the report at http://localhost:8001 and keeps the cell running until you interrupt the kernel.

**Note:** in Ryzen AI 1.7.1 the AI Analyzer supports BF16 models only ([AI Analyzer docs](https://ryzenai.docs.amd.com/en/1.7.1/ai_analyzer.html)). For the XINT8 model in this notebook no analysis files are generated, so the analyzer has nothing to show.

In [ ]:
!aianalyzer ./ -p 8001